# Parse Open Alex


In [ ]:
from __future__ import annotations

import os

import duckdb
import numpy as np
from tqdm import tqdm

In [ ]:
main_dir = os.path.dirname(os.getcwd())

extracted_database_path = os.path.join(main_dir, "data/inter/openalex_attempt.ddb")
medical_database_path = os.path.join(main_dir, "data/inter/medical_attempt.db")

In [ ]:
extracted_db = duckdb.connect(extracted_database_path, read_only=True)
medical_db = duckdb.connect(medical_database_path, read_only=False)

In [ ]:
medical_df = extracted_db.execute("""
    SELECT * FROM works WHERE field = 'Medicine'
""").fetchdf()

medical_db.execute("""
    CREATE or REPLACE TABLE works AS SELECT * FROM medical_df LIMIT 0
""")

medical_db.execute("""
    INSERT INTO works SELECT * FROM medical_df ORDER BY openalex_id
""")

del medical_df
extracted_db.close()

print(f"Total Works {medical_db.execute('SELECT COUNT(*) FROM works').fetchone()[0]}")

In [ ]:
medical_db.execute("""
    ALTER TABLE works ADD COLUMN IF NOT EXISTS matrix_id BIGINT
""")

medical_db.execute("""
    WITH NumberedWorks AS (
        SELECT openalex_id, row_number() OVER (ORDER BY openalex_id) AS rn
        FROM works
    )
    UPDATE works
    SET matrix_id = (
        SELECT rn FROM NumberedWorks
        WHERE NumberedWorks.openalex_id = works.openalex_id
    )
""")

In [ ]:
total_works = medical_db.execute("SELECT COUNT(*) FROM works").fetchone()[0]
print(f"Total Works {total_works:_d}")

total_refs = medical_db.execute("""
    SELECT COUNT(*)
    FROM works, UNNEST(referenced_works) AS all_refs
""").fetchone()[0]
print(f"Total References {total_refs:_d}")


medical_db.execute("""
    SELECT matrix_id, openalex_id, title FROM works LIMIT 10
""").fetchdf()

In [ ]:
openalex_id_array = medical_db.execute("""
    SELECT openalex_id FROM works
""").fetchnumpy()["openalex_id"]

ref_works_df = medical_db.execute("""
    SELECT matrix_id, referenced_works FROM works
""").fetchdf()

ref_array = np.zeros((total_refs, 2), dtype=np.int64)
i_ref = 0
for _, row in tqdm(
    ref_works_df.iterrows(), total=total_works, desc="Indexing Referenced Works"
):
    for ref in row["referenced_works"]:
        mat_id = np.searchsorted(openalex_id_array, ref)
        if mat_id == len(openalex_id_array) or openalex_id_array[mat_id] != ref:
            continue
        ref_array[i_ref, 0] = row["matrix_id"]
        ref_array[i_ref, 1] = mat_id + 1
        i_ref += 1


ref_array = ref_array[:i_ref]
print(f"Total Valid References {i_ref:_d}")

ref_array = np.concatenate([ref_array, ref_array[:, [1, 0]]])

print("Sorting References")
ref_array = ref_array[ref_array[:, 0].argsort()]